# Notebook for Synthetic Data Generation Exploration

For my first Substack series 😎

## Import packages

In [3]:
#import necessary libraries

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo 

## Data

### Import dataset

more info regarding the data at UCI ML [here](https://archive.ics.uci.edu/dataset/222/bank+marketing).

In [68]:
# Import dataset to work with
bank_marketing = fetch_ucirepo(id=222)
bank_df = bank_marketing.data.original
  
bank_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


### Simple EDA

In [5]:
print(bank_df.y.value_counts())
print('='*40)
print("Total percentage of users that did not convert for the campaign = ", round(bank_df.y.value_counts(normalize=True).iloc[0]*100.0, 2), "%")
print("Total percentage of users that did convert for the campaign = ", round(bank_df.y.value_counts(normalize=True).iloc[1]*100.0, 2), "%")

y
no     39922
yes     5289
Name: count, dtype: int64
Total percentage of users that did not convert for the campaign =  88.3 %
Total percentage of users that did convert for the campaign =  11.7 %


In [6]:
bank_df.describe()

,age,balance,day_of_week,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


Label is largely imbalanced.

Lets fit a simple model to see baseline performance then test other methods of synthetic data generation

### Preprocess Data

In [69]:
y = (bank_df['y'] == 'yes').astype(int)
X = bank_df.drop('y', axis=1)

In [70]:
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [71]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: y, dtype: int64

In [72]:
## Split data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=4,
    stratify=y)

In [73]:
X_train_encoded = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test, drop_first=True, dtype=int)


## Model Experiments

### RandomForest Model

In [74]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

#### 1. Baseline

In [75]:
rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=7,
    n_jobs=-1
)

In [76]:
rf_baseline.fit(X_train_encoded, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

#### 2. Traditional Fix i.e. weighted classes

In [77]:
rf_weighted = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=7,
    n_jobs=-1
)

In [78]:
rf_weighted.fit(X_train_encoded, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

#### 3. Upsample Minority Class with SMOTE 

In [79]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=11)
X_train_smote, y_train_smote = smote.fit_resample(X_train_encoded, y_train)


In [80]:
print("Total count of labels in original data:", y_train.value_counts())
print("Total count of labels in upsampled data:", y_train_smote.value_counts())
print('='*40)
print("Total percentage of users that did not convert for the campaign (SMOTE) = ", round(y_train_smote.value_counts(normalize=True).iloc[0]*100.0, 2), "%")
print("Total percentage of users that did convert for the campaign (SMOTE) = ", round(y_train_smote.value_counts(normalize=True).iloc[1]*100.0, 2), "%")

Total count of labels in original data: y
0    31937
1     4231
Name: count, dtype: int64
Total count of labels in upsampled data: y
0    31937
1    31937
Name: count, dtype: int64
Total percentage of users that did not convert for the campaign (SMOTE) =  50.0 %
Total percentage of users that did convert for the campaign (SMOTE) =  50.0 %


We now got our label distribution to a 50/50 split instead of 88/12 split we had previously

In [81]:
rf_smote = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=7,
    n_jobs=-1
)

In [82]:
rf_smote.fit(X_train_smote, y_train_smote)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

### CTGAN Upsampling

In [84]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
from sdv.sampling import Condition

In [85]:
# Build full training df with label
train_df = X_train.copy()
train_df['y'] = y_train.values

# Auto-detect metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train_df)

#### Fit the GAN Model

In [88]:
# Using a small epochs count of 100 to train reasonably fast
ctgan = CTGANSynthesizer(
    metadata,
    epochs=200,
    batch_size=500,
    generator_dim=(128, 128, 128),
    discriminator_dim=(128, 128, 128),
    verbose=True
)
ctgan.fit(train_df)


/Users/oscar.lares/Library/CloudStorage/OneDrive-CoxAutomotive/Documents/Personal/GH-Personal/Synthetic-Data-Generation/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/Users/oscar.lares/Library/CloudStorage/OneDrive-CoxAutomotive/Documents/Personal/GH-Personal/Synthetic-Data-Generation/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-1.37) | Discrim. (0.16): 100%|██████████| 200/200 [09:20<00:00,  2.80s/it] 


In [89]:
# Count how many minority samples you need
n_majority = (y_train == 0).sum()
n_minority = (y_train == 1).sum()
n_to_generate = n_majority - n_minority
print(n_to_generate)

27706


In [90]:
# Condition sampling on minority class
condition = Condition(num_rows=int(n_to_generate), column_values={'y': 1})
synthetic_minority = ctgan.sample_from_conditions(conditions=[condition])

Sampling conditions: 100%|██████████| 27706/27706 [00:04<00:00, 5960.00it/s]


In [94]:
# Reconstruct balanced training set
X_syn = synthetic_minority.drop(columns=['y'])
y_syn = synthetic_minority['y']

X_train_ctgan = pd.concat([X_train, X_syn], ignore_index=True)
y_train_ctgan = pd.concat([y_train, y_syn], ignore_index=True)

#one hot encode
X_train_ctgan_encoded = pd.get_dummies(X_train_ctgan, drop_first=True, dtype=int)

In [92]:
print("Total count of labels in original data:", y_train.value_counts())
print("Total count of labels in upsampled data (CTGAN):", y_train_ctgan.value_counts())
print('='*40)
print("Total percentage of users that did not convert for the campaign (CTGAN) = ", round(y_train_ctgan.value_counts(normalize=True).iloc[0]*100.0, 2), "%")
print("Total percentage of users that did convert for the campaign (CTGAN) = ", round(y_train_ctgan.value_counts(normalize=True).iloc[1]*100.0, 2), "%")

Total count of labels in original data: y
0    31937
1     4231
Name: count, dtype: int64
Total count of labels in upsampled data (CTGAN): y
0    31937
1    31937
Name: count, dtype: int64
Total percentage of users that did not convert for the campaign (CTGAN) =  50.0 %
Total percentage of users that did convert for the campaign (CTGAN) =  50.0 %


In [95]:
rf_ctgan = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=7,
    n_jobs=-1
)
rf_ctgan.fit(X_train_ctgan_encoded, y_train_ctgan)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

### Comparing RF Model Results

In [97]:
print("=== 1. Baseline ===")
print(classification_report(y_test, rf_baseline.predict(X_test_encoded)))

print("=== 2. Class Weight Balanced ===")
print(classification_report(y_test, rf_weighted.predict(X_test_encoded)))

print("=== 3. SMOTE Synthetic Oversampling ===")
print(classification_report(y_test, rf_smote.predict(X_test_encoded)))

print("=== 4. CTGAN Synthetic Oversampling ===")
print(classification_report(y_test, rf_ctgan.predict(X_test_encoded)))

=== 1. Baseline ===
              precision    recall  f1-score   support

           0       0.91      0.98      0.95      7985
           1       0.68      0.29      0.40      1058

    accuracy                           0.90      9043
   macro avg       0.80      0.63      0.67      9043
weighted avg       0.88      0.90      0.88      9043

=== 2. Class Weight Balanced ===
              precision    recall  f1-score   support

           0       0.98      0.85      0.91      7985
           1       0.43      0.84      0.57      1058

    accuracy                           0.85      9043
   macro avg       0.70      0.85      0.74      9043
weighted avg       0.91      0.85      0.87      9043

=== 3. SMOTE Synthetic Oversampling ===
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      7985
           1       0.54      0.59      0.57      1058

    accuracy                           0.89      9043
   macro avg       0.75      0.76  

In [98]:
synthetic_minority.describe()


,age,balance,day_of_week,duration,campaign,pdays,previous,y
count,27706.000000,27706.000000,27706.000000,27706.000000,27706.000000,27706.000000,27706.000000,27706.0
mean,41.123583,2600.948278,13.722190,495.418682,1.873493,80.284198,1.621057,1.0
std,12.680402,3224.596546,7.285269,345.886718,1.556418,97.116115,2.165174,0.0
min,18.000000,-916.000000,1.000000,0.000000,1.000000,-1.000000,0.000000,1.0
25%,32.000000,502.000000,8.000000,233.000000,1.000000,-1.000000,0.000000,1.0
50%,37.000000,1432.000000,13.000000,432.000000,2.000000,92.000000,1.000000,1.0
75%,52.000000,3790.000000,19.000000,658.000000,2.000000,119.000000,2.000000,1.0
max,91.000000,32719.000000,31.000000,2152.000000,28.000000,525.000000,12.000000,1.0


In [99]:
train_df[train_df['y']==1].describe()

,age,balance,day_of_week,duration,campaign,pdays,previous,y
count,4231.000000,4231.000000,4231.000000,4231.000000,4231.000000,4231.000000,4231.000000,4231.0
mean,41.604349,1834.602694,15.021744,535.654219,2.127629,68.047743,1.183408,1.0
std,13.411541,3648.804748,8.448929,392.084346,1.883930,117.541919,2.536433,0.0
min,18.000000,-3058.000000,1.000000,8.000000,1.000000,-1.000000,0.000000,1.0
25%,31.000000,214.000000,8.000000,242.000000,1.000000,-1.000000,0.000000,1.0
50%,38.000000,751.000000,15.000000,426.000000,2.000000,-1.000000,0.000000,1.0
75%,50.000000,2160.500000,21.000000,721.000000,2.000000,98.000000,2.000000,1.0
max,95.000000,81204.000000,31.000000,3253.000000,29.000000,854.000000,58.000000,1.0
